# ADNI_statiscs

In [ ]:
import pandas as pd
import numpy as np

fold_file_paths = [
    './0901/adni_merged_probs_fold0.csv',
    './0901/adni_merged_probs_fold1.csv',
    './0901/adni_merged_probs_fold2.csv',
    './0901/adni_merged_probs_fold3.csv',
    './0901/adni_merged_probs_fold4.csv'
]


print("⏳ 正在读取并合并 5 折数据...")
dfs = []
for path in fold_file_paths:
    try:
        df = pd.read_csv(path)
        dfs.append(df)
    except Exception as e:
        print(f"⚠️ 读取 {path} 失败: {e}")

if not dfs:
    raise ValueError("❌ 没有成功读取任何数据文件！")

merged_df = pd.concat(dfs, ignore_index=True)
print(f"✅ 合并完成！总样本数: {merged_df.shape[0]}, 总特征数: {merged_df.shape[1]}\n")

actual_features = [f for f in features if f in merged_df.columns]
missing_in_df = set(features) - set(actual_features)

if missing_in_df:
    print(f"⚠️ 警告: 有 {len(missing_in_df)} 个特征在 CSV 中不存在，已自动忽略。")

feature_groups = {}
for f in actual_features:
    prefix = f.split('_')[0] if '_' in f else 'other'
    if prefix not in feature_groups:
        feature_groups[prefix] = []
    feature_groups[prefix].append(f)

print("📊 ================= 类别缺失率统计 ================= 📊")
print(f"{'类别 (Prefix)':<15} | {'特征数量':<10} | {'整体缺失率':<12}")
print("-" * 45)

summary_results = []

for prefix, cols in feature_groups.items():
    subset = merged_df[cols]
    
    total_cells = subset.size
    missing_cells = subset.isnull().sum().sum()
    
    missing_rate = missing_cells / total_cells if total_cells > 0 else 0
    
    summary_results.append({
        'Category': prefix,
        'Feature_Count': len(cols),
        'Missing_Rate': missing_rate
    })
    
    print(f"{prefix:<15} | {len(cols):<10} | {missing_rate:.2%}")

global_subset = merged_df[actual_features]
global_missing_rate = global_subset.isnull().sum().sum() / global_subset.size

print("-" * 45)
print(f"{'🌟 全局汇总':<14} | {len(actual_features):<10} | {global_missing_rate:.2%}")
print("========================================================\n")

"""
print("🔍 详细查看各类别缺失最严重的 Top 3 特征:")
for prefix, cols in feature_groups.items():
    print(f"\n▶ [{prefix}] 类别:")
    rates = merged_df[cols].isnull().mean().sort_values(ascending=False)
    for col, rate in rates.head(3).items():
        if rate > 0:
            print(f"   - {col}: {rate:.2%}")
"""


In [ ]:
import pandas as pd

fold_file_paths = [
    './0901/adni_merged_probs_fold0.csv',
    './0901/adni_merged_probs_fold1.csv',
    './0901/adni_merged_probs_fold2.csv',
    './0901/adni_merged_probs_fold3.csv',
    './0901/adni_merged_probs_fold4.csv'
]

target_file_path = './dataset/adni/pre_csv/adni_final0929.csv'

target_data = pd.read_csv(target_file_path)

for i, fold_file_path in enumerate(fold_file_paths):
    fold_data = pd.read_csv(fold_file_path)
    
    if 'PTID' not in fold_data.columns:
        print(f"错误：文件 {fold_file_path} 中不存在 'PTID' 列。请检查文件。")
        continue
    
    fold_ptids = fold_data['PTID'].dropna()
    
    filtered_target_data = target_data[target_data['PTID'].isin(fold_ptids)]
    
    merged_data = fold_data.copy()
    
    for column in filtered_target_data.columns:
        if column in fold_data.columns:
            merged_data[column] = merged_data[column].combine_first(filtered_target_data[column])
    
    output_file_path = f'./0901/adni_merged_probs_fold{i}_updated.csv'
    
    merged_data.to_csv(output_file_path, index=False)
    
    print(f"更新后的五折文件 {output_file_path} 已保存")


In [ ]:
output_path = './0901/adni_merged_probs_fold_updated.csv'

adni_data= pd.read_csv(output_path)
race_counts = adni_data['his_RACE'].value_counts()

race_percentage = (race_counts / race_counts.sum()) * 100

print("\nRace distribution:")
for race, count in race_counts.items():
    print(f"{race}: {count} ({race_percentage[race]:.2f}%)")

In [ ]:
import os
import pandas as pd
from scipy import stats
import numpy as np

def format_p_value(p_value):
    if p_value < 1e-300:
        p_value = 1e-300
    return f"{p_value:.4e}"

labels = [
    "NC",
    "MCI",
    "DE"
]

race_mapping = {
    'whi': 'White',
    'blk': 'Black',
    'asi': 'Asian',
    'ind': 'American Indian',
    'oth': 'multirace',
    'haw': 'Pacific',
    'White': 'White',
    'Black': 'Black',
    'ASIAN': 'Asian',
    'more than one': 'multirace',
    'AIAN': 'American Indian'
}

adni_count = len(adni_data) 
print(f"ADNI 队列数量: {adni_count}")

adni_data['his_RACE_mapped'] = adni_data['his_RACE'].map(race_mapping).fillna('multirace')

adni_t1_exist_count = adni_data['t1_path'].dropna().apply(lambda x: os.path.exists(str(x))).sum()
adni_t2_exist_count = adni_data['t2_path'].dropna().apply(lambda x: os.path.exists(str(x))).sum()

print(f"ADNI 队列中 t1_path 文件存在的行数: {adni_t1_exist_count}")
print(f"ADNI 队列中 t2_path 文件存在的行数: {adni_t2_exist_count}")


adni_age_mean = adni_data['Visit_Age'].mean()
adni_age_std = adni_data['Visit_Age'].std()
print(f"ADNI Age: mean ± s.d. = {adni_age_mean:.2f} ± {adni_age_std:.2f}")

adni_sex_counts = adni_data['his_SEX'].value_counts()
adni_sex_percentage = (adni_sex_counts / adni_sex_counts.sum()) * 100
print("\nADNI Sex distribution:")
for sex, count in adni_sex_counts.items():
    print(f"{sex}: {count} ({adni_sex_percentage[sex]:.2f}%)")

adni_educ_mean = adni_data['his_EDUC'].mean()
adni_educ_std = adni_data['his_EDUC'].std()
print(f"\nADNI Education (years): mean ± s.d. = {adni_educ_mean:.2f} ± {adni_educ_std:.2f}")

adni_race_counts = adni_data['his_RACE_mapped'].value_counts()
adni_race_percentage = (adni_race_counts / adni_race_counts.sum()) * 100
print("\nADNI Race distribution:")
for race, count in adni_race_counts.items():
    print(f"{race}: {count} ({adni_race_percentage[race]:.2f}%)")

adni_cdr_counts = adni_data['cdr_CDRGLOB'].value_counts()
adni_cdr_percentage = (adni_cdr_counts / adni_cdr_counts.sum()) * 100
print("\nADNI CDR distribution:")
for cdr, count in adni_cdr_counts.items():
    print(f"{cdr}: {count} ({adni_cdr_percentage[cdr]:.2f}%)")

print("\nADNI 标签分布情况 (每个标签列的 value_counts):")
for target in labels:
    if target in adni_data.columns:
        counts = adni_data[target].value_counts()
        total = counts.sum()
        percentage = (counts / total) * 100 if total > 0 else pd.Series()
        print(f"标签: {target}")
        for val, count in counts.items():
            print(f"  {val}: {count} ({percentage.get(val, 0):.2f}%)")
        print(f"  总计: {total}")
        print("-" * 30)
    else:
        print(f"警告: 列 '{target}' 在 ADNI 数据中不存在，跳过。")

print("\n总结统计 (表格格式):")
race_categories = ['White', 'Black', 'Asian', 'American Indian', 'Pacific', 'multirace']

for target in labels:
    if target not in adni_data.columns:
        print(f"警告: 列 '{target}' 不存在，跳过 {target} 的总结。")
        continue
    
    adni_subset = adni_data[adni_data[target] == 1]
    if adni_subset.empty:
        print(f"ADNI_{target}\t无数据")
        continue
    
    adni_total = len(adni_subset)
    adni_age_mean = adni_subset['Visit_Age'].mean()
    adni_age_std = adni_subset['Visit_Age'].std()
    adni_male_count = (adni_subset['his_SEX'] == 'male').sum()
    adni_male_pct = (adni_male_count / adni_total * 100) if adni_total > 0 else 0
    adni_educ_mean = adni_subset['his_EDUC'].mean()
    adni_educ_std = adni_subset['his_EDUC'].std()
    adni_race_n = [adni_subset['his_RACE_mapped'].value_counts().get(cat, 0) for cat in race_categories]
    adni_cdr_mean = adni_subset['cdr_CDRGLOB'].mean()
    adni_cdr_std = adni_subset['cdr_CDRGLOB'].std()
    
    adni_t1_count = adni_subset['t1_path'].dropna().apply(lambda x: os.path.exists(str(x))).sum()
    adni_t2_count = adni_subset['t2_path'].dropna().apply(lambda x: os.path.exists(str(x))).sum()

    print(f"ADNI_{target}\t"
        f"{adni_age_mean:.2f} ± {adni_age_std:.2f}\t"
        f"{adni_male_count} ({adni_male_pct:.2f}%)\t"
        f"{adni_educ_mean:.2f} ± {adni_educ_std:.2f}\t"
        f"{', '.join(map(str, adni_race_n))}\t"
        f"{adni_cdr_mean:.2f} ± {adni_cdr_std:.2f}\t"
        f"{adni_t1_count}, {adni_t2_count}")


print("\n组间 P 值计算 (针对 ADNI 队列的所有非空诊断标签子集):")

subsets = {}
for target in labels:
    if target in adni_data.columns:
        subset = adni_data[adni_data[target] == 1]
        if not subset.empty:
            subsets[target] = subset

if len(subsets) < 2:
    print("不足 2 个非空子集，无法计算 P 值。")
else:
    age_groups = [subset['Visit_Age'].dropna().values for subset in subsets.values()]
    educ_groups = [subset['his_EDUC'].dropna().values for subset in subsets.values()]
    cdr_groups = [subset['cdr_CDRGLOB'].dropna().values for subset in subsets.values()]

    age_p = stats.f_oneway(*age_groups).pvalue if all(len(g) > 0 for g in age_groups) else np.nan
    educ_p = stats.f_oneway(*educ_groups).pvalue if all(len(g) > 0 for g in educ_groups) else np.nan
    cdr_p = stats.f_oneway(*cdr_groups).pvalue if all(len(g) > 0 for g in cdr_groups) else np.nan

    male_table = np.array([
        [(subset['his_SEX'] == 'male').sum(), len(subset) - (subset['his_SEX'] == 'male').sum()]
        for subset in subsets.values()
    ])
    if np.any(male_table):
        _, male_p, _, _ = stats.chi2_contingency(male_table)
    else:
        male_p = np.nan

    race_table = np.array([
        [subset['his_RACE_mapped'].value_counts().get(cat, 0) for cat in race_categories]
        for subset in subsets.values()
    ])
    race_table = race_table[:, np.any(race_table != 0, axis=0)]
    race_table = race_table[np.any(race_table != 0, axis=1), :]
    if np.any(race_table):
        non_zero_rows = np.all(race_table > 0, axis=1)
        non_zero_cols = np.all(race_table > 0, axis=0)
        filtered_race_table = race_table[non_zero_rows][:, non_zero_cols]

        if filtered_race_table.size > 0 and np.any(filtered_race_table):
            _, race_p, _, _ = stats.chi2_contingency(filtered_race_table)
        else:
            race_p = np.nan
    else:
        race_p = np.nan

    print(f"Age P-value (ANOVA): {format_p_value(age_p)} {'(显著差异)' if age_p < 0.05 else '(无显著差异)'}")
    print(f"Male P-value (Chi-square): {format_p_value(male_p)} {'(显著差异)' if male_p < 0.05 else '(无显著差异)'}")
    print(f"Education P-value (ANOVA): {format_p_value(educ_p)} {'(显著差异)' if educ_p < 0.05 else '(无显著差异)'}")
    print(f"Race P-value (Chi-square): {format_p_value(race_p)} {'(显著差异)' if race_p < 0.05 else '(无显著差异)'}")
    print(f"CDR P-value (ANOVA): {format_p_value(cdr_p)} {'(显著差异)' if cdr_p < 0.05 else '(无显著差异)'}")


In [ ]:
import numpy as np
from scipy.stats import chi2_contingency

data = np.array([[119, 4, 2, 0, 0, 129],
                 [107, 2, 1, 1, 0, 107],
                 [109, 5, 3, 0, 0, 117]])

merged_data = np.column_stack((data[:, :3], data[:, 3:5].sum(axis=1)))

chi2, p, dof, expected = chi2_contingency(merged_data)
print(f"Chi2: {chi2}, P-value: {p}, DOF: {dof}")



# output

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_recall_curve, roc_auc_score
import os
def find_optimal_threshold(y_true, y_probs, step=0.01, metric='f1'):
    thresholds = np.arange(0.0, 1.0 + step, step)
    best_thr, best_score = 0.5, -1.0

    for thr in thresholds:
        preds = (y_probs >= thr).astype(int)
        if   metric == 'acc':
            score = accuracy_score(y_true, preds)
        elif metric == 'f1':
            score = f1_score(y_true, preds, zero_division=0)
        elif metric == 'precision':
            score = precision_score(y_true, preds, zero_division=0)
        elif metric == 'recall':
            score = recall_score(y_true, preds, zero_division=0)
        elif metric == 'auc':
            score = roc_auc_score(y_true, y_probs)
        elif metric == 'prauc':
            score = average_precision_score(y_true, y_probs) 
        else:
            raise ValueError("Unsupported metric")

        if score > best_score:
            best_score, best_thr = score, thr

    return best_thr, best_score


def process_multiclass_predictions(y_probs_group):
    n_samples, n_classes = y_probs_group.shape
    y_pred_group = np.zeros_like(y_probs_group)
    
    max_indices = np.argmax(y_probs_group, axis=1)
    
    for sample_idx in range(n_samples):
        if np.max(y_probs_group[sample_idx]) > 0:
            y_pred_group[sample_idx, max_indices[sample_idx]] = 1
    
    return y_pred_group

def add_final_labels_to_csvs(base_dir='./0901/', 
                             optimization_metric='f1', 
                             output_suffix='_with_final_labels'):
    
    multilabel_names = ['AD', 'LBD', 'VD', 'diagnosis_FTD', 'EXC', 'PSY', 'ODE']
    multilabel_prob_cols = [f'{name}_y' for name in multilabel_names]
    multilabel_label_cols = [f'{name}_x' for name in multilabel_names]
    multilabel_final_cols = [f'{name}_final' for name in multilabel_names]

    group1_names = ['NC', 'MCI', 'DE']
    group1_prob_cols = [f'{name}_y' for name in group1_names]
    group1_label_cols = [f'{name}_x' for name in group1_names]
    group1_final_cols = [f'{name}_final' for name in group1_names]

    group2_names = ['MCI_A', 'MCI_AM', 'CI', 'MCI_Na', 'MCI_NaM', 'non_MCI']
    group2_prob_cols = [f'{name}_y' for name in group2_names]
    group2_label_cols = [f'{name}_x' for name in group2_names]
    group2_final_cols = [f'{name}_final' for name in group2_names]

    all_final_cols = group1_final_cols + group2_final_cols + multilabel_final_cols

    for fold_idx in range(5):
        file_path = os.path.join(base_dir, f'merged_probs_fold{fold_idx}.csv')
        output_path = os.path.join(base_dir, f'merged_probs_fold{fold_idx}{output_suffix}.csv')
        
        if not os.path.exists(file_path):
            print(f"❌ 文件不存在: {file_path}")
            continue
        
        print(f"\n处理 Fold {fold_idx}: {file_path}")
        
        df = pd.read_csv(file_path)
        if 'MCI_nonA' in df.columns:
            df['non_MCI_y'] = df['MCI_nonA']
            print(f"Fold {fold_idx} - 创建MCI_nonA列，复制自MCI_nonA")
        else:
            print(f"Fold {fold_idx} - 警告：MCI_nonA列不存在")
        
        y_probs_multilabel = df[multilabel_prob_cols].values
        y_true_multilabel = df[multilabel_label_cols].values
        
        y_pred_multilabel = np.zeros_like(y_true_multilabel)
        fold_thresholds = {}
        
        print(f"Fold {fold_idx} - 为七标签寻找最佳阈值 (优化指标: {optimization_metric.upper()}):")
        print("-" * 70)
        
        for label_idx, label_name in enumerate(multilabel_names):
            y_true_label = y_true_multilabel[:, label_idx]
            y_prob_label = y_probs_multilabel[:, label_idx]
            
            unique_labels = np.unique(y_true_label)
            if len(unique_labels) < 2:
                print(f"{label_name:<15}: 只有一个类别 ({unique_labels}), 使用默认阈值0.5")
                optimal_threshold = 0.5
                best_score = 0.0
            else:
                optimal_threshold, best_score = find_optimal_threshold(
                    y_true_label, y_prob_label, 
                    step=0.01, metric=optimization_metric
                )
            
            y_pred_multilabel[:, label_idx] = (y_prob_label >= optimal_threshold).astype(int)
            
            fold_thresholds[label_name] = {
                'threshold': optimal_threshold,
                f'{optimization_metric}_score': best_score,
                'positive_samples': int(y_true_label.sum()),
                'total_samples': len(y_true_label),
                'positive_ratio': float(y_true_label.sum() / len(y_true_label))
            }
            
            print(f"{label_name:<15}: 阈值={optimal_threshold:.3f}, "
                  f"{optimization_metric.upper()}={best_score:.3f}, "
                  f"正样本={int(y_true_label.sum())}/{len(y_true_label)} "
                  f"({y_true_label.sum()/len(y_true_label)*100:.1f}%)")
        
        print("-" * 70)
        
        for col_idx, final_col in enumerate(multilabel_final_cols):
            df[final_col] = y_pred_multilabel[:, col_idx]
        
        print(f"Fold {fold_idx} - 处理Group1 多分类: {group1_names}")
        y_probs_group1 = df[group1_prob_cols].values
        y_pred_group1 = process_multiclass_predictions(y_probs_group1)
        
        for col_idx, final_col in enumerate(group1_final_cols):
            df[final_col] = y_pred_group1[:, col_idx]
        
        print(f"Fold {fold_idx} - 处理Group2 多分类: {group2_names}")
        y_probs_group2 = df[group2_prob_cols].values
        y_pred_group2 = process_multiclass_predictions(y_probs_group2)
        
        for col_idx, final_col in enumerate(group2_final_cols):
            df[final_col] = y_pred_group2[:, col_idx]
        
        df.to_csv(output_path, index=False)
        print(f"✅ 已保存更新文件: {output_path}")
        print(f"  添加的列: {all_final_cols}")

if __name__ == "__main__":
    add_final_labels_to_csvs()


# result3

In [ ]:
import re
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional, Tuple, Union

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['mathtext.bf'] = 'Arial:bold'

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams.update({
    'font.size': 7,
    'axes.titlesize': 8,
    'axes.labelsize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 6,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5,
})

np.random.seed(42)

RESULTS_ROOT = Path("./adRAG/ADNI_results")
creative_day = '20250906'
PLOT_DIR = Path(f'./pic/{creative_day}')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ["Claude-Sonnet-4", "DeepSeek-R1", "GPT-5"]

MODEL_PALETTE = {
    "Claude-Sonnet-4": "#4C72B0",
    "DeepSeek-R1": "#55A868",
    "GPT-5": "#C44E52",
}

METRIC_KEYS = [
    ("Accuracy", "acc"),
    ("Precision (Macro)", "precision"),
    ("Recall (Macro)", "recall"),
    ("F1-score (Macro)", "f1"),
    ("MCC", "MCC"),
    ("Specificity (Macro)", "specificity"),
]
METRIC_ORDER = [display for _, display in METRIC_KEYS]


def parse_metric_entry(entry: str) -> Tuple[float, Optional[float]]:
    match = re.match(r"\s*([0-9]*\.?[0-9]+)\s*(?:\(([^)]+)\))?", str(entry))
    if match:
        value = float(match.group(1))
        baseline = float(match.group(2)) if match.group(2) else None
        return value, baseline
    value = float(entry)
    return value, None


def load_fold_metrics(fold_index: int) -> pd.DataFrame:
    metrics_path = RESULTS_ROOT / f"model-fold{fold_index}" / "primary_label_metrics.json"
    with metrics_path.open() as f:
        metrics_payload = json.load(f)

    rows = []
    for model_name, metric_block in metrics_payload.items():
        for metric_key, metric_display in METRIC_KEYS:
            raw_value = metric_block.get(metric_key)
            if raw_value is None:
                continue
            value, baseline = parse_metric_entry(raw_value)
            rows.append(
                {
                    "fold": fold_index,
                    "model": model_name,
                    "metric_key": metric_key,
                    "metric_display": metric_display,
                    "value": value,
                    "baseline_value": baseline,
                }
            )
    return pd.DataFrame(rows)


def load_model_without_label_metrics() -> pd.DataFrame:
    metrics_path = RESULTS_ROOT / "model-without-label" / "primary_label_metrics.json"
    with metrics_path.open() as f:
        metrics_payload = json.load(f)

    repeat_entries = [
        metrics
        for name, metrics in metrics_payload.items()
        if name.startswith("Claude-Sonnet-4-repeat") or name == "Claude-Sonnet-4"
    ]
    
    if not repeat_entries:
        raise KeyError("No Claude-Sonnet-4 metrics found in model-without-label results.")

    rows = []
    for metric_key, metric_display in METRIC_KEYS:
        values = []
        for metric_block in repeat_entries:
            raw_value = metric_block.get(metric_key)
            if raw_value is None:
                continue
            try:
                numeric_value = float(raw_value)
            except (TypeError, ValueError):
                numeric_value, _ = parse_metric_entry(raw_value)
            values.append(numeric_value)

        if not values:
            continue

        mean_value = float(np.mean(values))
        std_value = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

        rows.append(
            {
                "metric_key": metric_key,
                "metric_display": metric_display,
                "mean_value": mean_value,
                "std_value": std_value,
                "repeat_count": len(values),
            }
        )

    return pd.DataFrame(rows)

 
print("="*100)
print("Loading data...")
print("="*100)

fold_metrics_list = []
for fold_idx in range(5):
    try:
        fold_data = load_fold_metrics(fold_idx)
        fold_metrics_list.append(fold_data)
        print(f"✓ Fold {fold_idx} loaded: {len(fold_data)} records")
    except FileNotFoundError:
        print(f"⚠ Warning: Fold {fold_idx} not found, skipping...")
        continue

if not fold_metrics_list:
    raise ValueError("No fold data found!")

fold_metrics = pd.concat(fold_metrics_list, ignore_index=True)
print(f"\n✓ Total fold metrics loaded: {len(fold_metrics)} records")

baseline_per_fold = (
    fold_metrics.dropna(subset=["baseline_value"])
    .groupby(["fold", "metric_display"], as_index=False)["baseline_value"]
    .first()
)
print(f"✓ Baseline data extracted: {len(baseline_per_fold)} records")

combined_summary = (
    fold_metrics[fold_metrics["model"] == "Claude-Sonnet-4"]
    .groupby("metric_display")
    .agg(mean_value=("value", "mean"), std_value=("value", "std"))
    .reset_index()
)
print(f"✓ Claude-Sonnet-4+Classifier summary computed: {len(combined_summary)} metrics")

baseline_summary = (
    baseline_per_fold.groupby("metric_display")
    .agg(mean_value=("baseline_value", "mean"), std_value=("baseline_value", "std"))
    .reset_index()
)
print(f"✓ Classifier summary computed: {len(baseline_summary)} metrics")

try:
    pure_llm = load_model_without_label_metrics()
    print(f"✓ Pure LLM data loaded: {len(pure_llm)} metrics")
    print(f"  Columns: {pure_llm.columns.tolist()}")
except Exception as e:
    print(f"✗ Error loading pure LLM data: {e}")
    raise

pure_llm_summary = pure_llm[["metric_display", "mean_value", "std_value"]].copy()

bars_df = pd.concat(
    [
        pure_llm_summary.assign(group="Claude-Sonnet-4"),
        combined_summary.assign(group="Claude-Sonnet-4+Classifier"),
        baseline_summary.assign(group="Classifier"),
    ],
    ignore_index=True,
)

print(f"\n✓ Combined data shape: {bars_df.shape}")
print(f"✓ Groups: {bars_df['group'].unique().tolist()}")
print(f"✓ Metrics: {bars_df['metric_display'].unique().tolist()}")

bars_df["metric_display"] = pd.Categorical(
    bars_df["metric_display"], categories=METRIC_ORDER, ordered=True
)

GROUP_ORDER = ["Claude-Sonnet-4+Classifier", "Classifier","Claude-Sonnet-4"]
GROUP_PALETTE = {
    "Claude-Sonnet-4": "#8172B2",
    "Classifier": "#8C8C8C",
    "Claude-Sonnet-4+Classifier": "#4C72B0",
}

bars_df["std_value"] = bars_df["std_value"].fillna(0.0)

print("\n" + "="*100)
print("Data completeness check:")
print("="*100)
for group in GROUP_ORDER:
    group_data = bars_df[bars_df['group'] == group]
    print(f"\n{group}:")
    print(f"  Records: {len(group_data)}")
    print(f"  Metrics covered: {group_data['metric_display'].tolist()}")

fig_width = 7.2
fig_height = 5.0
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

y = np.arange(len(METRIC_ORDER))
height = 0.22

means_dict = {}
stds_dict = {}
for group in GROUP_ORDER:
    means_dict[group] = []
    stds_dict[group] = []
    for metric in METRIC_ORDER:
        row = bars_df[(bars_df['metric_display'] == metric) & (bars_df['group'] == group)]
        if len(row) > 0:
            means_dict[group].append(row['mean_value'].values[0])
            stds_dict[group].append(row['std_value'].values[0])
        else:
            print(f"⚠ Warning: Missing data for {group} - {metric}")
            means_dict[group].append(0)
            stds_dict[group].append(0)

print("\n" + "="*100)
print("Generating plot...")
print("="*100)

for i, group in enumerate(GROUP_ORDER):
    offset = (i - 1) * height
    bars = ax.barh(
        y + offset, 
        means_dict[group], 
        height, 
        xerr=stds_dict[group], 
        color=GROUP_PALETTE[group], 
        alpha=0.85, 
        label=group, 
        capsize=2.5,
        error_kw={'linewidth': 0.8, 'elinewidth': 0.8},
        edgecolor='black', 
        linewidth=0.5
    )
    
    for j, (mean_val, std_val) in enumerate(zip(means_dict[group], stds_dict[group])):
        if mean_val > 0:
            text_x = mean_val + std_val + 0.015
            ax.text(
                text_x, 
                y[j] + offset, 
                f'{mean_val:.3f}±{std_val:.3f}',
                va='center', 
                ha='left', 
                fontsize=5.5, 
                color='black'
            )

ax.set_xlim(0.4, 1.05)
ax.set_yticks(y)
ax.set_yticklabels(METRIC_ORDER, fontsize=7, fontweight='normal')
ax.set_xlabel('Score', fontsize=7)
ax.set_ylabel('Metric', fontsize=7)

ax.grid(True, alpha=0.2, linestyle='-', axis='x', linewidth=0.3)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.legend(
    loc='lower right', 
    fontsize=6, 
    frameon=True, 
    framealpha=1.0, 
    edgecolor='black', 
    fancybox=False
)

plt.tight_layout()

output_path = PLOT_DIR / "model_group_barplot.pdf"
plt.savefig(output_path, dpi=1200, bbox_inches='tight', format='pdf', backend='pdf')
print(f"\n✓ Figure saved to: {output_path}")
plt.show()

print("\n" + "="*100)
print("PERFORMANCE COMPARISON (Mean ± Std)")
print("="*100)

for i, metric in enumerate(METRIC_ORDER):
    print(f"\n{'─'*100}")
    print(f"{metric}:")
    print(f"{'─'*100}")
    
    for group in GROUP_ORDER:
        mean_val = means_dict[group][i]
        std_val = stds_dict[group][i]
        print(f"  {group:35s}: {mean_val:.4f} ± {std_val:.4f}")
    
    classifier_mean = means_dict['Classifier'][i]
    llm_mean = means_dict['Claude-Sonnet-4'][i]
    llm_plus_mean = means_dict['Claude-Sonnet-4+Classifier'][i]
    
    if classifier_mean > 0:
        llm_improvement = ((llm_mean - classifier_mean) / classifier_mean) * 100
        llm_plus_improvement = ((llm_plus_mean - classifier_mean) / classifier_mean) * 100
        
        print(f"\n  {'Relative Improvement vs Classifier:'}")
        print(f"    Claude-Sonnet-4:              {llm_improvement:+6.2f}%")
        print(f"    Claude-Sonnet-4+Classifier:   {llm_plus_improvement:+6.2f}%")

print("\n" + "="*100)

print("\nAVERAGE IMPROVEMENT ACROSS ALL METRICS:")
print("="*100)

avg_llm_improvement = 0
avg_llm_plus_improvement = 0
count = 0

for i in range(len(METRIC_ORDER)):
    classifier_mean = means_dict['Classifier'][i]
    if classifier_mean > 0:
        llm_improvement = ((means_dict['Claude-Sonnet-4'][i] - classifier_mean) / classifier_mean) * 100
        llm_plus_improvement = ((means_dict['Claude-Sonnet-4+Classifier'][i] - classifier_mean) / classifier_mean) * 100
        avg_llm_improvement += llm_improvement
        avg_llm_plus_improvement += llm_plus_improvement
        count += 1

if count > 0:
    avg_llm_improvement /= count
    avg_llm_plus_improvement /= count
    print(f"  Claude-Sonnet-4 vs Classifier:              {avg_llm_improvement:+6.2f}%")
    print(f"  Claude-Sonnet-4+Classifier vs Classifier:   {avg_llm_plus_improvement:+6.2f}%")

print("="*100)

comparison_data = []
for i, metric in enumerate(METRIC_ORDER):
    row = {'Metric': metric}
    for group in GROUP_ORDER:
        mean_val = means_dict[group][i]
        std_val = stds_dict[group][i]
        row[group] = f"{mean_val:.4f}±{std_val:.4f}"
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
csv_path = PLOT_DIR / "metrics_comparison_table.csv"
comparison_df.to_csv(csv_path, index=False)
print(f"\n✓ Comparison table saved to: {csv_path}")

print("\n" + "="*100)
print("✓ ALL TASKS COMPLETED SUCCESSFULLY!")
print("="*100)


In [ ]:
def parse_metric_entry(entry):
    match = re.match(r"\s*([0-9]*\.?[0-9]+)\s*(?:\(([^)]+)\))?", str(entry))
    if match:
        value = float(match.group(1))
        baseline = float(match.group(2)) if match.group(2) else None
        return value, baseline
    value = float(entry)
    return value, None


def load_fold_metrics(fold_index):
    metrics_path = RESULTS_ROOT / f"model-fold{fold_index}" / "primary_label_metrics.json"
    with metrics_path.open() as f:
        metrics_payload = json.load(f)

    rows = []
    for model_name, metric_block in metrics_payload.items():
        for metric_key, metric_display in METRIC_KEYS:
            raw_value = metric_block.get(metric_key)
            if raw_value is None:
                continue
            value, baseline = parse_metric_entry(raw_value)
            rows.append({
                "fold": fold_index,
                "model": model_name,
                "metric_key": metric_key,
                "metric_display": metric_display,
                "value": value,
                "baseline_value": baseline,
            })
    return pd.DataFrame(rows)


fold_metrics_list = []
for fold_idx in range(5):
    try:
        fold_data = load_fold_metrics(fold_idx)
        fold_metrics_list.append(fold_data)
    except FileNotFoundError:
        print(f"Warning: Fold {fold_idx} not found, skipping...")
        continue

if not fold_metrics_list:
    raise ValueError("No fold data found!")

fold_metrics = pd.concat(fold_metrics_list, ignore_index=True)

fold_metrics["metric_display"] = pd.Categorical(
    fold_metrics["metric_display"], 
    categories=METRIC_ORDER, 
    ordered=True
)

box_data = fold_metrics.dropna(subset=["metric_display", "value"])

fig_width = 7.2
fig_height = 5.0
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

positions = np.arange(len(METRIC_ORDER))

box_props = dict(linewidth=0.8)
whisker_props = dict(linewidth=0.8)
cap_props = dict(linewidth=0.8)
median_props = dict(linewidth=1.2, color='black')
mean_props = dict(marker='_', markerfacecolor='black', markeredgecolor='black', 
                 markersize=8, linewidth=1.2)

for i, metric_name in enumerate(METRIC_ORDER):
    metric_data = box_data[box_data['metric_display'] == metric_name]
    
    for j, model_name in enumerate(MODEL_ORDER):
        model_values = metric_data[metric_data['model'] == model_name]['value'].values
        if len(model_values) > 0:
            offset = (j - 1) * 0.25
            pos = i + offset
            
            box = ax.boxplot(
                model_values, 
                positions=[pos], 
                widths=0.2,
                patch_artist=True, 
                vert=False, 
                showmeans=True,
                boxprops=box_props, 
                whiskerprops=whisker_props,
                capprops=cap_props, 
                medianprops=median_props,
                meanprops=mean_props
            )
            
            for b in box['boxes']:
                b.set_facecolor(MODEL_PALETTE[model_name])
                b.set_alpha(0.6)
                b.set_edgecolor('black')
            
            ax.scatter(
                model_values, 
                [pos] * len(model_values),
                color=MODEL_PALETTE[model_name], 
                alpha=0.5, 
                s=15, 
                edgecolors='black', 
                linewidth=0.3,
                zorder=3
            )

summary_stats = (
    box_data.groupby(["metric_display", "model"])
    .agg(mean_value=("value", "mean"), std_value=("value", "std"))
    .reset_index()
)
summary_stats["std_value"].fillna(0.0, inplace=True)

ax.set_yticks(positions)
ax.set_yticklabels(METRIC_ORDER, fontsize=7, fontweight='normal')
ax.set_xlabel('Score', fontsize=7)
ax.set_ylabel('Metric', fontsize=7)
ax.set_xlim(0.5, 0.9)

ax.grid(True, alpha=0.2, linestyle='-', axis='x', linewidth=0.3)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_elements = [
    Patch(facecolor=MODEL_PALETTE[model], alpha=0.6, edgecolor='black', 
          linewidth=0.5, label=model)
    for model in MODEL_ORDER
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=6, 
          frameon=True, framealpha=1.0, edgecolor='black', fancybox=False)

plt.tight_layout()

output_path = PLOT_DIR / "model_metrics_boxplot.pdf"
plt.savefig(output_path, dpi=1200, bbox_inches='tight', format='pdf', backend='pdf')
print(f"Figure saved to: {output_path}")
plt.show()

print("\n性能对比 (均值 ± 标准差):")
print("=" * 80)
for metric in METRIC_ORDER:
    print(f"\n{metric}:")
    metric_stats = summary_stats[summary_stats['metric_display'] == metric]
    for model in MODEL_ORDER:
        model_stat = metric_stats[metric_stats['model'] == model]
        if not model_stat.empty:
            mean_val = model_stat['mean_value'].values[0]
            std_val = model_stat['std_value'].values[0]
            print(f"  {model:20s}: {mean_val:.4f} ± {std_val:.4f}")


# multimodal compaison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

colors = ['#E64B35', '#4DBBD5', '#7E6148'] 
markers = ['o', 's', '^']
models = ['Multimodal', 'Tabular Only', 'Image Only']

raw_data = {
    'Multimodal': {
        'exact_match': [0.5884, 0.0058],
        'hamming_loss': [0.1574, 0.0044],
        'jaccard_score': [0.6847, 0.0053],
        'precision_macro': [0.2811, 0.0032],
        'recall_macro': [0.8589, 0.0131],
        'f1_macro': [0.3473, 0.0028]
    },
    'Tabular Only': {
        'exact_match': [0.5218, 0.0206],
        'hamming_loss': [0.2168, 0.0179],
        'jaccard_score': [0.6303, 0.0195],
        'precision_macro': [0.2473, 0.0099],
        'recall_macro': [0.9481, 0.0045],
        'f1_macro': [0.3212, 0.0110]
    },
    'Image Only': {
        'exact_match': [0.0025, 0.0030],
        'hamming_loss': [0.4721, 0.0380],
        'jaccard_score': [0.1667, 0.0158],
        'precision_macro': [0.1181, 0.0013],
        'recall_macro': [0.6735, 0.0418],
        'f1_macro': [0.1548, 0.0030]
    }
}

metric_keys = ['exact_match', 'hamming_loss', 'jaccard_score', 'precision_macro', 'recall_macro', 'f1_macro']
display_labels = ['Exact Match', 'Hamming\nAccuracy', 'Jaccard', 'Precision', 'Recall', 'F1']

processed_data = {}
for model in models:
    processed_data[model] = {'means': [], 'stds': []}
    for key in metric_keys:
        val_mean = raw_data[model][key][0]
        val_std = raw_data[model][key][1]
        if key == 'hamming_loss':
            processed_data[model]['means'].append(1 - val_mean)
            processed_data[model]['stds'].append(val_std)
        else:
            processed_data[model]['means'].append(val_mean)
            processed_data[model]['stds'].append(val_std)

N = len(metric_keys)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig = plt.figure(figsize=(15, 6.5), dpi=1000)
gs = GridSpec(1, 2, width_ratios=[1, 1.3], wspace=0.15)

ax = fig.add_subplot(gs[0], projection='polar')
ax.set_facecolor('#FAFAFA')

performance_rings = [0.2, 0.4, 0.6, 0.8, 1.0]
ring_colors = ['#FFE5E5', '#FFF4E5', '#E5F7E5', '#E5F0FF', '#F9F9F9']

for i, (level, color) in enumerate(zip(performance_rings, ring_colors)):
    lower = performance_rings[i-1] if i > 0 else 0
    ax.fill_between(np.linspace(0, 2*np.pi, 100), lower, level, color=color, alpha=0.5, zorder=0)

for i, model in enumerate(models):
    values = processed_data[model]['means']
    values_plot = values + [values[0]]
    c = colors[i]
    m = markers[i]
    
    ax.plot(angles, values_plot, linewidth=2.5, color=c, label=model, zorder=i+5, clip_on=False)
    ax.fill(angles, values_plot, color=c, alpha=0.1, zorder=i+4)
    ax.scatter(angles, values_plot, color=c, marker=m, s=20, 
               edgecolors='white', linewidth=1.2, zorder=i+10, clip_on=False)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(display_labels, fontsize=11, fontweight='bold', color='#2C3E50')

ax.set_ylim(0, 1.0)
ax.set_yticks(performance_rings)
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], 
                   fontsize=9, color='#444444', fontweight='bold')

ax.grid(False)
grid_color = '#BCBCBC'

for r in performance_rings:
    if r == 1.0:
        linestyle = '-'
        linewidth = 1.2
        alpha = 1.0
        color = '#999999'
    else:
        linestyle = '--'
        linewidth = 0.8
        alpha = 0.8
        color = grid_color
        
    ax.plot(np.linspace(0, 2*np.pi, 200), [r]*200, 
            color=color, linestyle=linestyle, linewidth=linewidth, alpha=alpha, zorder=1)

for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 1.0], 
            color=grid_color, linestyle='--', linewidth=0.8, alpha=0.8, zorder=1)

ax.spines['polar'].set_visible(False) 

ax_table = fig.add_subplot(gs[1])
ax_table.axis('off')

col_labels = ['Metric'] + models
table_cells = []

for idx, label in enumerate(display_labels):
    row = [label]
    for model in models:
        mean = processed_data[model]['means'][idx]
        std = processed_data[model]['stds'][idx]
        row.append(f"{mean:.4f} ± {std:.4f}")
    table_cells.append(row)

table = ax_table.table(
    cellText=table_cells,
    colLabels=col_labels,
    loc='center',
    cellLoc='center',
    colWidths=[0.2] + [0.25]*3,
    edges='open'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.2)

heavy_linewidth = 1.5
light_linewidth = 0.8

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(0)
    cell.set_text_props(fontfamily='Arial')
    if row == 0:
        cell.set_text_props(weight='bold', color='black')
        cell.set_edgecolor('black')
        cell.set_linewidth(light_linewidth)
        cell.visible_edges = 'B'
    else:
        if col > 0:
            val_str = table_cells[row-1][col]
            current_val = float(val_str.split(' ±')[0])
            row_vals = [float(table_cells[row-1][c].split(' ±')[0]) for c in range(1, 4)]
            if current_val == max(row_vals):
                cell.set_text_props(weight='bold', color='black')
    if row == len(table_cells):
        cell.set_edgecolor('black')
        cell.set_linewidth(heavy_linewidth)
        cell.visible_edges = 'B'

ax_table.add_line(Line2D([0, 1], [0.75, 0.75], transform=ax_table.transAxes, 
                         color='black', linewidth=heavy_linewidth))


handles = [Line2D([0], [0], color=c, marker=m, linestyle='-', linewidth=2, markersize=8, markerfacecolor=c, markeredgecolor='white') for c, m in zip(colors, markers)]
fig.legend(handles, models, loc='lower center', bbox_to_anchor=(0.5, 0.02), 
           ncol=3, frameon=False, fontsize=11)

plt.subplots_adjust(bottom=0.15)
save_path = 'Nature_Radar_SolidOuterLine.pdf'
plt.savefig(save_path, dpi=1200, bbox_inches='tight', transparent=True)
print(f"✅ 最终定稿: 最外圈实线，内圈虚线: {save_path}")

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['mathtext.bf'] = 'Arial:bold'

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams.update({
    'font.size': 7,
    'axes.titlesize': 8,
    'axes.labelsize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 6,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
})

raw_data = {
    'mHC': {
        'Exact Match': (0.659, 0.004),
        'Global Hamming': (0.920, 0.012),
        'F1-Macro': (0.347, 0.006),
        'AUC-Macro': (0.889, 0.015),
        'Subset1': (0.897, 0.006),
        'Subset2': (0.804, 0.007),
        'Subset3': (0.899, 0.027)
    },
    'Multimodal JLS': {
        'Exact Match': (0.588, 0.006),
        'Global Hamming': (0.843, 0.004),
        'F1-Macro': (0.347, 0.003),
        'AUC-Macro': (0.907, 0.003),
        'Subset1': (0.879, 0.008),
        'Subset2': (0.720, 0.005),
        'Subset3': (0.834, 0.007)
    },
    'Multimodal Transformer': {
        'Exact Match': (0.588, 0.005),
        'Global Hamming': (0.849, 0.002),
        'F1-Macro': (0.347, 0.002),
        'AUC-Macro': (0.900, 0.006),
        'Subset1': (0.856, 0.008),
        'Subset2': (0.691, 0.009),
        'Subset3': (0.778, 0.005)
    },
    'Multimodal Catboost': {
        'Exact Match': (0.519, 0.006),
        'Global Hamming': (0.811, 0.005),
        'F1-Macro': (0.329, 0.004),
        'AUC-Macro': (0.925, 0.005),
        'Subset1': (0.772, 0.011),
        'Subset2': (0.537, 0.025),
        'Subset3': (0.833, 0.008)
    },
    'Tabular Classifier': {
        'Exact Match': (0.522, 0.021),
        'Global Hamming': (0.783, 0.018),
        'F1-Macro': (0.321, 0.011),
        'AUC-Macro': (0.918, 0.004),
        'Subset1': (0.783, 0.028),
        'Subset2': (0.586, 0.030),
        'Subset3': (0.782, 0.020)
    },
    'Image Classifier': {
        'Exact Match': (0.003, 0.003),
        'Global Hamming': (0.528, 0.038),
        'F1-Macro': (0.155, 0.003),
        'AUC-Macro': (0.629, 0.010),
        'Subset1': (0.295, 0.030),
        'Subset2': (0.106, 0.056),
        'Subset3': (0.574, 0.047)
    }
}

GROUP_ORDER = [
    "mHC",
    "Multimodal JLS",
    "Multimodal Transformer",
    "Multimodal Catboost",
    "Tabular Classifier",
    "Image Classifier"
]

GROUP_PALETTE = {
    "mHC": "#3C5488",
    "Multimodal JLS": "#4DBBD5",
    "Multimodal Transformer": "#E64B35",
    "Multimodal Catboost": "#00A087",
    "Tabular Classifier": "#F39B7F",
    "Image Classifier": "#8491B4"
}


panel_a_keys = ['Exact Match', 'Global Hamming', 'F1-Macro', 'AUC-Macro']
panel_a_labels = ['Exact Match', 'Hamming Acc\n(Global)', 'F1-Macro', 'AUC-Macro']

panel_b_keys = ['Subset1', 'Subset2', 'Subset3']
panel_b_labels = ['Subset1\n(Overall Acc)', 'Subset2\n(Overall Acc)', 'Subset3\n(Hamming Acc)']

def get_plot_data(keys_list):
    means = {group: [] for group in GROUP_ORDER}
    stds = {group: [] for group in GROUP_ORDER}
    for key in keys_list:
        for group in GROUP_ORDER:
            mean, std = raw_data[group][key]
            means[group].append(mean)
            stds[group].append(std)
    return means, stds

a_means, a_stds = get_plot_data(panel_a_keys)
b_means, b_stds = get_plot_data(panel_b_keys)

fig_width = 7.2
fig_height = 5.0
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(fig_width, fig_height))

n_groups = len(GROUP_ORDER)
total_width = 0.85
bar_height = total_width / n_groups

def plot_panel(ax, metric_labels, means_dict, stds_dict, title):
    y_pos = np.arange(len(metric_labels))
    
    for i, group in enumerate(GROUP_ORDER):
        offset = (i - n_groups/2 + 0.5) * bar_height
        
        ax.barh(
            y_pos + offset,
            means_dict[group],
            bar_height,
            xerr=stds_dict[group],
            color=GROUP_PALETTE[group],
            alpha=0.9,
            label=group,
            capsize=1.2,
            error_kw={'linewidth': 0.5, 'elinewidth': 0.5},
            edgecolor='white',
            linewidth=0.4
        )
        
        for j, (mean, std) in enumerate(zip(means_dict[group], stds_dict[group])):
            text_x = mean + std + 0.015
            label_text = f'{mean:.4f}±{std:.4f}'
            ax.text(
                text_x, y_pos[j] + offset, label_text,
                va='center', ha='left',
                fontsize=4,
                color='black',
                fontweight='normal'
            )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(metric_labels, fontsize=7)
    ax.set_xlabel('Score', fontsize=7)
    ax.set_title(title, fontsize=8, fontweight='bold', loc='left', pad=10)
    
    ax.grid(True, alpha=0.2, linestyle='-', axis='x', linewidth=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.set_xlim(0, 1.35)

plot_panel(ax1, panel_a_labels, a_means, a_stds, 'A  Global Performance Metrics')

plot_panel(ax2, panel_b_labels, b_means, b_stds, 'B  Subset Performance')

handles, labels = ax1.get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.02),
    ncol=5,
    frameon=False,
    fontsize=7
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)

save_dir = Path('./')
save_dir.mkdir(parents=True, exist_ok=True)
output_pdf = save_dir / "Comparison_AllLabels.pdf"
plt.savefig(output_pdf, dpi=1200, bbox_inches='tight', stransparent=True)

print(f"✅ 图表已生成 (含所有数值): {output_pdf}")
plt.show()


# heatmap

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, matthews_corrcoef, roc_curve, precision_recall_curve, auc
)
from sklearn.preprocessing import label_binarize
import os

creative_day = '20250906'
pic_dir = f'./pic/{creative_day}/adni_text_report'
os.makedirs(pic_dir, exist_ok=True)

def load_fold_data(fold_idx):
    file_path = f'./0901/adni_merged_probs_fold{fold_idx}.csv'
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"文件未找到: {file_path}")

    df = pd.read_csv(file_path)
    
    prob_cols = ['NC_prob', 'MCI_prob', 'DE_prob']
    label_cols = ['NC', 'MCI', 'DE']
    
    y_probs = df[prob_cols].values
    y_true_onehot = df[label_cols].values
    
    y_true = np.argmax(y_true_onehot, axis=1)
    y_pred = np.argmax(y_probs, axis=1)
    
    return y_true, y_pred, y_probs

def evaluate_fold(y_true, y_pred, y_probs, fold_idx):
    
    class_names = ['NC', 'MCI', 'DE']
    n_classes = len(class_names)
    
    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    mcc_overall = matthews_corrcoef(y_true, y_pred)
    
    specificities = []
    mccs_per_class = []
    roc_auc_per_class = []
    pr_auc_per_class = []
    
    y_true_bin = label_binarize(y_true, classes=range(n_classes))
    
    for i in range(n_classes):
        y_true_i = (y_true == i).astype(int)
        y_pred_i = (y_pred == i).astype(int)
        
        tn, fp, fn, tp = confusion_matrix(y_true_i, y_pred_i).ravel()
        
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)
        
        mcc_i = matthews_corrcoef(y_true_i, y_pred_i)
        mccs_per_class.append(mcc_i)
        
        try:
            roc_auc = roc_auc_score(y_true_bin[:, i], y_probs[:, i])
        except:
            roc_auc = 0.5
        roc_auc_per_class.append(roc_auc)
        
        try:
            pr_auc = average_precision_score(y_true_bin[:, i], y_probs[:, i])
        except:
            pr_auc = 0.0
        pr_auc_per_class.append(pr_auc)

    unique, counts = np.unique(y_true, return_counts=True)
    class_supports = np.zeros(n_classes)
    for idx, count in zip(unique, counts):
        if idx < n_classes: class_supports[idx] = count
            
    spec_macro = np.mean(specificities)
    spec_weighted = np.average(specificities, weights=class_supports)

    try:
        auc_macro = roc_auc_score(y_true_bin, y_probs, average='macro', multi_class='ovr')
        auc_micro = roc_auc_score(y_true_bin, y_probs, average='micro', multi_class='ovr')
    except:
        auc_macro, auc_micro = 0.5, 0.5
        
    try:
        aupr_macro = np.mean(pr_auc_per_class)
        aupr_micro = average_precision_score(y_true_bin, y_probs, average='micro')
    except:
        aupr_macro, aupr_micro = 0.0, 0.0

    print(f"Fold {fold_idx + 1} | Acc: {acc:.4f} | MCC: {mcc_overall:.4f} | Spec(Macro): {spec_macro:.4f} | AUC(Macro): {auc_macro:.4f}")
    
    return {
        'accuracy': acc, 
        'mcc_overall': mcc_overall,
        'specificity_macro': spec_macro,
        'specificity_weighted': spec_weighted,
        'precision_macro': prec_macro, 
        'recall_macro': rec_macro,
        'f1_macro': f1_macro, 
        'roc_auc_macro': auc_macro, 
        'roc_auc_micro': auc_micro, 
        'pr_auc_macro': aupr_macro, 
        'pr_auc_micro': aupr_micro,
        
        'roc_auc_per_class': roc_auc_per_class,
        'pr_auc_per_class': pr_auc_per_class,
        'specificities_per_class': specificities,
        'mccs_per_class': mccs_per_class
    }

def main():
    print("🚀 开始 NC-MCI-DE 纯参数评估 (无绘图)...")
    print(f"📂 结果输出目录: {pic_dir}")
    
    fold_results = []
    
    for fold_idx in range(5):
        try:
            y_true, y_pred, y_probs = load_fold_data(fold_idx)
            result = evaluate_fold(y_true, y_pred, y_probs, fold_idx)
            fold_results.append(result)
        except Exception as e:
            print(f"❌ Fold {fold_idx + 1} 失败: {e}")
            continue
    
    if not fold_results:
        print("没有成功完成任何 Fold 的评估。")
        return

    print("\n" + "="*50)
    print("📊 5-Fold Cross-Validation 最终统计")
    print("="*50)

    metrics_map = {
        'Accuracy': 'accuracy',
        'MCC (Overall)': 'mcc_overall',
        'Specificity (Macro)': 'specificity_macro',
        'Specificity (Weighted)': 'specificity_weighted',
        'Precision (Macro)': 'precision_macro',
        'Recall (Macro)': 'recall_macro',
        'F1-Score (Macro)': 'f1_macro',
        'ROC-AUC (Macro)': 'roc_auc_macro',
        'PR-AUC (Macro)': 'pr_auc_macro'
    }

    print(f"{'Metric':<25} | {'Mean ± Std':<20}")
    print("-" * 50)
    
    for display_name, key in metrics_map.items():
        values = [r[key] for r in fold_results]
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"{display_name:<25} | {mean_val:.4f} ± {std_val:.4f}")

    print("\n" + "="*50)
    print("🧩 各类别详细统计 (Per-Class Metrics)")
    print("="*50)
    
    class_names = ['NC', 'MCI', 'DE']
    metrics_per_class = [
        ('ROC-AUC', 'roc_auc_per_class'),
        ('PR-AUC', 'pr_auc_per_class'),
        ('Specificity', 'specificities_per_class'),
        ('MCC', 'mccs_per_class')
    ]

    header = f"{'Class':<6} | " + " | ".join([f"{m[0]:<15}" for m in metrics_per_class])
    print(header)
    print("-" * len(header))

    for i, cls_name in enumerate(class_names):
        row_str = f"{cls_name:<6} | "
        for _, key in metrics_per_class:
            values = [r[key][i] for r in fold_results]
            mean_val = np.mean(values)
            std_val = np.std(values)
            row_str += f"{mean_val:.4f}±{std_val:.4f}   | "
        print(row_str)

    results_df = pd.DataFrame()
    
    for d_name, key in metrics_map.items():
        vals = [r[key] for r in fold_results]
        results_df = pd.concat([results_df, pd.DataFrame({
            'Metric': [d_name],
            'Mean': [np.mean(vals)], 'Std': [np.std(vals)],
            'Fold_1': [vals[0]], 'Fold_2': [vals[1]], 'Fold_3': [vals[2]], 'Fold_4': [vals[3]], 'Fold_5': [vals[4]]
        })], ignore_index=True)
    
    for i, cls_name in enumerate(class_names):
        for m_name, key in metrics_per_class:
            vals = [r[key][i] for r in fold_results]
            results_df = pd.concat([results_df, pd.DataFrame({
                'Metric': [f'{cls_name} - {m_name}'],
                'Mean': [np.mean(vals)], 'Std': [np.std(vals)],
                'Fold_1': [vals[0]], 'Fold_2': [vals[1]], 'Fold_3': [vals[2]], 'Fold_4': [vals[3]], 'Fold_5': [vals[4]]
            })], ignore_index=True)

    for col in results_df.columns:
        if col != 'Metric': results_df[col] = results_df[col].round(4)
        
    csv_path = f'{pic_dir}/NC_MCI_DE_metrics_report.csv'
    results_df.to_csv(csv_path, index=False)
    print(f"\n✅ 详细参数表已保存至: {csv_path}")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from mpl_toolkits.axes_grid1 import make_axes_locatable

output_dir = './pic/comparison_heatmap'
os.makedirs(output_dir, exist_ok=True)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['axes.unicode_minus'] = False

def plot_nature_style_heatmap_v5():
    metrics = ['Specificity', 'MCC', 'F1-Score', 'Accuracy']
    
    models =  ['mHC','Transformer','CatBoost']

    means = np.array([
        [ 0.8501, 0.8367,0.8062],
        [ 0.5926, 0.5518,0.4289],
        [ 0.6358, 0.6125,0.5517],
        [ 0.7062, 0.6791,0.6131]
    ])

    stds = np.array([
        [0.0206, 0.0181, 0.0160],
        [0.0622, 0.0400, 0.0397],
        [0.0344, 0.0577, 0.0382],
        [0.0415, 0.0339, 0.0355]
    ])

    annot_matrix = np.empty_like(means, dtype=object)
    for i in range(means.shape[0]):
        for j in range(means.shape[1]):
            annot_matrix[i, j] = f"{means[i, j]:.3f}\n±{stds[i, j]:.3f}"

    fig, ax = plt.subplots(figsize=(5.0, 5.5))

    heatmap = sns.heatmap(means, 
                annot=annot_matrix, 
                fmt='',             
                cmap='YlGnBu',      
                vmin=0.0, vmax=1.0, 
                cbar=False,
                linewidths=1.5,     
                linecolor='white',  
                square=True,        
                annot_kws={"size": 13, "weight": "normal"}, 
                ax=ax)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    
    cbar = plt.colorbar(heatmap.get_children()[0], cax=cax)
    cbar.set_label('Performance Score', fontsize=12, weight='bold')
    cbar.set_ticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    cbar.ax.tick_params(labelsize=10)

    ax.xaxis.tick_bottom()
    ax.set_xticklabels(models, fontsize=12, fontweight='bold')
    ax.set_yticklabels(metrics, fontsize=12, fontweight='bold', rotation=0)
    
    ax.tick_params(axis='both', which='both', length=0)

    plt.tight_layout()
    
    pdf_path = os.path.join(output_dir, 'Model_Comparison_Heatmap.pdf')
    png_path = os.path.join(output_dir, 'Model_Comparison_Heatmap.png')
    
    plt.savefig(pdf_path, dpi=1000, bbox_inches='tight', format='pdf')
    plt.savefig(png_path, dpi=600, bbox_inches='tight', format='png')
    
    print(f"✅ 热力图已生成 (包含 Transformer)!")
    print(f"   PDF: {pdf_path}")
    print(f"   PNG: {png_path}")
    plt.show()

if __name__ == "__main__":
    plot_nature_style_heatmap_v5()


# sankey

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
import matplotlib.patches as patches
import os

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams.update({
    'font.size': 7,
    'axes.titlesize': 8,
    'axes.labelsize': 7,
    'figure.dpi': 300
})

llm_base_path = "./ADNI_results/model-fold{}/Claude-Sonnet-4/prediction_labels.csv"
mhc_base_path = "./0901/adni_merged_probs_fold{}.csv"
output_dir = "./pic/20250906"
os.makedirs(output_dir, exist_ok=True)
output_pdf_name = "Nature_Sankey_Mean_Wide.pdf"

LABELS = ['NC', 'MCI', 'DE']
COLOR_PALETTE = {
    'NC':  '#009E73',
    'MCI': '#E69F00',
    'DE':  '#D55E00'
}

print("Loading data...")
all_data = []
for fold_idx in range(5):
    p_llm = llm_base_path.format(fold_idx)
    p_mhc = mhc_base_path.format(fold_idx)
    
    if not os.path.exists(p_llm) or not os.path.exists(p_mhc): continue

    df_llm = pd.read_csv(p_llm)[['ID', 'primary_label']].rename(columns={'primary_label': 'Pred_Claude'})
    df_llm['ID'] = df_llm['ID'].astype(str)
    df_mhc = pd.read_csv(p_mhc)
    
    def get_true(row):
        if row['NC']==1: return 'NC'
        if row['MCI']==1: return 'MCI'
        if row['DE']==1: return 'DE'
        return None
    
    def get_pred(row):
        probs = {'NC': row['NC_prob'], 'MCI': row['MCI_prob'], 'DE': row['DE_prob']}
        return max(probs, key=probs.get)

    df_mhc['True_Label'] = df_mhc.apply(get_true, axis=1)
    df_mhc['Pred_mHC'] = df_mhc.apply(get_pred, axis=1)
    df_mhc['PTID'] = df_mhc['PTID'].astype(str)
    
    merged = pd.merge(df_llm, df_mhc[['PTID', 'True_Label', 'Pred_mHC']], 
                      left_on='ID', right_on='PTID', how='inner')
    all_data.append(merged)

df_final = pd.concat(all_data, ignore_index=True)
print(f"Data loaded. Total samples (Sum of 5 folds): {len(df_final)}")

def draw_sigmoid_link(ax, x1, x2, y1_top, y1_bot, y2_top, y2_bot, color, alpha=0.4):
    mid_x = (x1 + x2) / 2
    verts = [
        (x1, y1_bot),
        (mid_x, y1_bot), (mid_x, y2_bot), (x2, y2_bot),
        (x2, y2_top),
        (mid_x, y2_top), (mid_x, y1_top), (x1, y1_top),
        (x1, y1_bot)
    ]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.LINETO, Path.LINETO, Path.CURVE4, Path.CURVE4, Path.LINETO, Path.CLOSEPOLY]
    path = Path(verts, codes)
    patch = patches.PathPatch(path, facecolor=color, edgecolor='none', alpha=alpha)
    ax.add_patch(patch)

def plot_mean_sankey(df, filename):
    fig_w, fig_h = 9.5, 3.2 
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    
    cols = ['True_Label', 'Pred_mHC', 'Pred_Claude']
    col_names = ['Ground Truth', 'Baseline (mHC)', 'Claude Enhanced']
    x_coords = [0, 1.2, 2.4] 
    
    bar_width = 0.05 
    
    FOLD_COUNT = 5
    
    stage_counts = []
    for col in cols:
        counts = {label: len(df[df[col] == label]) / FOLD_COUNT for label in LABELS}
        stage_counts.append(counts)
    
    total_mean_count = len(df) / FOLD_COUNT
    gap_ratio = 0.08 
    gap_size = total_mean_count * gap_ratio
    
    y_positions = [{} for _ in range(3)]
    for i in range(3):
        col_total_h = sum(stage_counts[i].values()) + gap_size * (len(LABELS) - 1)
        current_y = col_total_h 
        
        for label in LABELS:
            count = stage_counts[i][label]
            top = current_y
            bot = current_y - count
            y_positions[i][label] = {'top': top, 'bot': bot, 'count': count}
            current_y = bot - gap_size

    cursors_out = [{l: y_positions[i][l]['top'] for l in LABELS} for i in range(2)]
    cursors_in  = [{l: y_positions[i][l]['top'] for l in LABELS} for i in range(1, 3)]

    for layer_idx in range(2):
        src_col = cols[layer_idx]
        tgt_col = cols[layer_idx+1]
        
        for src_label in LABELS:
            for tgt_label in LABELS:
                raw_flow = len(df[(df[src_col] == src_label) & (df[tgt_col] == tgt_label)])
                flow = raw_flow / FOLD_COUNT
                
                if flow > 0:
                    y1_top = cursors_out[layer_idx][src_label]
                    y1_bot = y1_top - flow
                    cursors_out[layer_idx][src_label] -= flow
                    
                    y2_top = cursors_in[layer_idx][tgt_label]
                    y2_bot = y2_top - flow
                    cursors_in[layer_idx][tgt_label] -= flow
                    
                    draw_sigmoid_link(
                        ax, 
                        x1=x_coords[layer_idx] + bar_width/2, 
                        x2=x_coords[layer_idx+1] - bar_width/2, 
                        y1_top=y1_top, y1_bot=y1_bot,
                        y2_top=y2_top, y2_bot=y2_bot,
                        color=COLOR_PALETTE[src_label],
                        alpha=0.35
                    )

    for i in range(3):
        for label in LABELS:
            pos = y_positions[i][label]
            height = pos['top'] - pos['bot']
            if height <= 0: continue
            
            rect = patches.Rectangle(
                (x_coords[i] - bar_width/2, pos['bot']),
                bar_width, height,
                facecolor=COLOR_PALETTE[label],
                edgecolor='white', linewidth=0.5, alpha=0.9
            )
            ax.add_patch(rect)
            
            text_y = (pos['top'] + pos['bot']) / 2
            
            if i == 0:
                ax.text(x_coords[i] - bar_width/2 - 0.05, text_y, label, 
                        ha='right', va='center', fontsize=8, fontweight='bold', color=COLOR_PALETTE[label])
            elif i == 2:
                ax.text(x_coords[i] + bar_width/2 + 0.05, text_y, label, 
                        ha='left', va='center', fontsize=8, fontweight='bold', color=COLOR_PALETTE[label])
            else:
                ax.text(x_coords[i], text_y, label, 
                        ha='center', va='center', fontsize=6, color='white', fontweight='bold')

    global_max_y = max([y_positions[0]['NC']['top'], y_positions[1]['NC']['top'], y_positions[2]['NC']['top']])
    
    for i, name in enumerate(col_names):
        ax.text(x_coords[i], global_max_y * 1.08, name, 
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xlim(x_coords[0]-0.5, x_coords[2]+0.5)
    ax.set_ylim(0, global_max_y * 1.2)
    ax.axis('off')
    
    plt.tight_layout()
    
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, format='pdf', bbox_inches='tight', dpi=1200)
    print(f"✓ PDF saved successfully: {os.path.abspath(save_path)}")

plot_mean_sankey(df_final, output_pdf_name)


# ADNI high\low

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import interp
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

PIC_DIR = "./results/plots_comparison"
os.makedirs(PIC_DIR, exist_ok=True)

MODELS = {
    "mHC": {
        "path_pattern": './0901/adni_merged_probs_fold{i}_updated.csv',
        "color": '#d62728',
        "linestyle": '-'
    },
    "Transformer": {
        "path_pattern": './results/0901/transformer_probs_fold{i}.csv',
        "color": '#2ca02c',
        "linestyle": '-'
    },
        "CatBoost": {
        "path_pattern": './results/0901/catboost_probs_fold{i}.csv',
        "color": '#1f77b4',
        "linestyle": '-'
    },
}

def load_fold_data(file_pattern, fold_idx):
    path = file_pattern.format(i=fold_idx)
    if not os.path.exists(path):
        path = file_pattern.format(i=fold_idx + 1)
        if not os.path.exists(path):
            return None, None

    try:
        df = pd.read_csv(path)
        
        if all(c in df.columns for c in ['NC_prob', 'MCI_prob', 'DE_prob']):
            y_probs = df[['NC_prob', 'MCI_prob', 'DE_prob']].values
        else:
            prob_cols = [c for c in df.columns if 'prob' in c.lower()]
            prob_cols.sort(key=lambda x: 0 if 'NC' in x else (1 if 'MCI' in x else 2))
            if len(prob_cols) != 3: return None, None
            y_probs = df[prob_cols].values

        y_true = None
        
        if 'True_Label_Str' in df.columns:
            label_map = {'NC': 0, 'MCI': 1, 'DE': 2}
            df['True_Label_Str'] = df['True_Label_Str'].astype(str).str.strip()
            y_true = df['True_Label_Str'].map(label_map).values
            
        elif 'True_Label_Idx' in df.columns:
            y_true = df['True_Label_Idx'].values
            
        elif all(c in df.columns for c in ['NC', 'MCI', 'DE']):
            y_true = np.argmax(df[['NC', 'MCI', 'DE']].values, axis=1)
            
        elif 'label' in df.columns:
            y_true = df['label'].values

        if y_true is not None:
            mask = ~pd.isna(y_true)
            y_true = y_true[mask].astype(int)
            y_probs = y_probs[mask]

        return y_true, y_probs
        
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None, None

def get_model_curves(model_name, config):
    print(f"\n🔄 Processing {model_name}...")
    
    fold_macro_fprs = []
    fold_macro_tprs = []
    fold_macro_aucs = []
    
    fold_macro_recalls = []
    fold_macro_precisions = []
    fold_macro_aps = []
    
    mean_fpr_grid = np.linspace(0, 1, 100)
    mean_recall_grid = np.linspace(0, 1, 100)
    
    valid_folds = 0
    
    for i in range(5):
        y_true, y_probs = load_fold_data(config['path_pattern'], i)
        if y_true is None: continue
        valid_folds += 1
        
        y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
        n_classes = 3
        
        fpr = dict()
        tpr = dict()
        for c in range(n_classes):
            fpr[c], tpr[c], _ = roc_curve(y_true_bin[:, c], y_probs[:, c])
            
        all_fpr = np.unique(np.concatenate([fpr[c] for c in range(n_classes)]))
        mean_tpr = np.zeros_like(all_fpr)
        for c in range(n_classes):
            mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])
        mean_tpr /= n_classes
        
        tpr_interp = np.interp(mean_fpr_grid, all_fpr, mean_tpr)
        tpr_interp[0] = 0.0
        fold_macro_tprs.append(tpr_interp)
        fold_macro_aucs.append(auc(all_fpr, mean_tpr))
        
        precision = dict()
        recall = dict()
        fold_class_precisions = []
        
        for c in range(n_classes):
            precision[c], recall[c], _ = precision_recall_curve(y_true_bin[:, c], y_probs[:, c])
            if len(recall[c]) > 1 and len(precision[c]) > 1:
                recall_rev = recall[c][::-1]
                precision_rev = precision[c][::-1]
                unique_recall, unique_indices = np.unique(recall_rev, return_index=True)
                unique_precision = precision_rev[unique_indices]
                
                if len(unique_recall) > 1:
                    p_interp = np.interp(mean_recall_grid, unique_recall, unique_precision)
                    fold_class_precisions.append(p_interp)

        if len(fold_class_precisions) >= 1:
            fold_macro_prec = np.mean(fold_class_precisions, axis=0)
            fold_macro_precisions.append(fold_macro_prec)
            fold_macro_aps.append(average_precision_score(y_true_bin, y_probs, average='macro'))

    print(f"   ✅ Processed {valid_folds} folds.")
    
    return {
        'roc_tprs': fold_macro_tprs, 'roc_aucs': fold_macro_aucs,
        'pr_precs': fold_macro_precisions, 'pr_aps': fold_macro_aps,
        'config': config
    }

def plot_comparison():
    results = {name: get_model_curves(name, cfg) for name, cfg in MODELS.items()}
    
    plt.figure(figsize=(6, 6))
    mean_fpr = np.linspace(0, 1, 100)
    
    for name, res in results.items():
        if not res['roc_tprs']: continue
        
        mean_tpr = np.mean(res['roc_tprs'], axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = np.mean(res['roc_aucs'])
        std_auc = np.std(res['roc_aucs'])
        
        color = res['config']['color']
        
        plt.plot(mean_fpr, mean_tpr, color=color, lw=3, linestyle='-',
                 label=f'{name} (AUC = {mean_auc:.3f} $\pm$ {std_auc:.3f})')
        
        std_tpr = np.std(res['roc_tprs'], axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color=color, alpha=0.15)

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PIC_DIR, 'Comparison_ROC_Macro.pdf'), dpi=1000, bbox_inches='tight')
    plt.show()
    plt.close()
    print("📄 Saved ROC Comparison.")

    plt.figure(figsize=(6, 6))
    mean_recall = np.linspace(0, 1, 100)
    
    for name, res in results.items():
        if not res['pr_precs']: continue
        
        mean_prec = np.mean(res['pr_precs'], axis=0)
        mean_ap = np.mean(res['pr_aps'])
        std_ap = np.std(res['pr_aps'])
        
        color = res['config']['color']
        
        plt.plot(mean_recall, mean_prec, color=color, lw=3, linestyle='-',
                 label=f'{name} (AP = {mean_ap:.3f} $\pm$ {std_ap:.3f})')
        
        std_prec = np.std(res['pr_precs'], axis=0)
        prec_upper = np.minimum(mean_prec + std_prec, 1)
        prec_lower = np.maximum(mean_prec - std_prec, 0)
        plt.fill_between(mean_recall, prec_lower, prec_upper, color=color, alpha=0.15)

    plt.axhline(y=0.333, color='k', linestyle='-.', alpha=0.7, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=14)
    plt.ylabel('Precision', fontsize=14)
    plt.legend(loc="lower left", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PIC_DIR, 'Comparison_PR_Macro.pdf'), dpi=1000, bbox_inches='tight')
    plt.show()
    plt.close()
    print("📄 Saved PR Comparison.")

if __name__ == "__main__":
    plot_comparison()


# mask plot

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['mathtext.bf'] = 'Arial:bold'

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams.update({
    'font.size': 7,
    'axes.titlesize': 8,
    'axes.labelsize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 7,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
})

raw_data = {
    'Mask Ratio = 0': {
        'Transformer': {
            'F1-macro': (0.2773, 0.0310),
            'AUC-macro': (0.7443, 0.0358),
            'AUPR-macro': (0.3017, 0.0228),
            'subset1 F1': (0.5746, 0.0232),
            'subset2 F1': (0.0441, 0.0418),
            'subset3 F1': (0.3164, 0.0281)
        },
        'mHC': {
            'F1-macro': (0.2867, 0.0254),
            'AUC-macro': (0.7418, 0.0287),
            'AUPR-macro': (0.2968, 0.0262),
            'subset1 F1': (0.5797, 0.0377),
            'subset2 F1': (0.0637, 0.0350),
            'subset3 F1': (0.3204, 0.0223)
        },
        'mHC + RAG-LLM': {
            'F1-macro': (0.3378, 0.0285),
            'AUC-macro': (0.8274, 0.0136),
            'AUPR-macro': (0.3897, 0.0289),
            'subset1 F1': (0.7391, 0.0447),
            'subset2 F1': (0.0864, 0.0648),
            'subset3 F1': (0.3454, 0.0305)
        }
    },

    'Mask Ratio = 0.3': {
        'Transformer': {
            'F1-macro': (0.349, 0.024),
            'AUC-macro': (0.831, 0.005),
            'AUPR-macro': (0.367, 0.026),
            'subset1 F1': (0.748, 0.025),
            'subset2 F1': (0.108, 0.036),
            'subset3 F1': (0.346, 0.019)
        },
        'mHC': {
            'F1-macro': (0.358, 0.031),
            'AUC-macro': (0.819, 0.032),
            'AUPR-macro': (0.389, 0.018),
            'subset1 F1': (0.767, 0.013),
            'subset2 F1': (0.140, 0.041),
            'subset3 F1': (0.339, 0.041)
        },
        'mHC + RAG-LLM': {
            'F1-macro': (0.371, 0.023),
            'AUC-macro': (0.822, 0.022),
            'AUPR-macro': (0.402, 0.011),
            'subset1 F1': (0.774, 0.023),
            'subset2 F1': (0.143, 0.064),
            'subset3 F1': (0.361, 0.014)
        }
    },

    'Mask Ratio = 0.6': {
        'Transformer': {
            'F1-macro': (0.287, 0.016),
            'AUC-macro': (0.752, 0.021),
            'AUPR-macro': (0.315, 0.018),
            'subset1 F1': (0.606, 0.036),
            'subset2 F1': (0.069, 0.012),
            'subset3 F1': (0.307, 0.025)
        },
        'mHC': {
            'F1-macro': (0.290, 0.049),
            'AUC-macro': (0.743, 0.021),
            'AUPR-macro': (0.336, 0.017),
            'subset1 F1': (0.632, 0.019),
            'subset2 F1': (0.111, 0.036),
            'subset3 F1': (0.271, 0.088)
        },
        'mHC + RAG-LLM': {
            'F1-macro': (0.322, 0.020),
            'AUC-macro': (0.751, 0.028),
            'AUPR-macro': (0.346, 0.021),
            'subset1 F1': (0.708, 0.025),
            'subset2 F1': (0.106, 0.032),
            'subset3 F1': (0.311, 0.031)
        }
    },

    'Mask Ratio = 0.9': {
        'Transformer': {
            'F1-macro': (0.105, 0.040),
            'AUC-macro': (0.583, 0.028),
            'AUPR-macro': (0.215, 0.030),
            'subset1 F1': (0.323, 0.029),
            'subset2 F1': (0.018, 0.024),
            'subset3 F1': (0.043, 0.069)
        },
        'mHC': {
            'F1-macro': (0.095, 0.010),
            'AUC-macro': (0.594, 0.024),
            'AUPR-macro': (0.218, 0.018),
            'subset1 F1': (0.317, 0.019),
            'subset2 F1': (0.026, 0.016),
            'subset3 F1': (0.049, 0.028)
        },
        'mHC + RAG-LLM': {
            'F1-macro': (0.161, 0.020),
            'AUC-macro': (0.665, 0.038),
            'AUPR-macro': (0.254, 0.018),
            'subset1 F1': (0.608, 0.039),
            'subset2 F1': (0.034, 0.017),
            'subset3 F1': (0.061, 0.033)
        }
    }
}


GROUP_ORDER = ["Transformer", "mHC", "mHC + RAG-LLM"]

GROUP_PALETTE = {
    "Transformer": "#8491B4",
    "mHC": "#3C5488",
    "mHC + RAG-LLM": "#E64B35"
}

panel_a_keys = ['F1-macro', 'AUC-macro', 'AUPR-macro']
panel_a_labels = ['F1-Macro', 'AUC-Macro', 'AUPR-Macro']

panel_b_keys = ['subset1 F1', 'subset2 F1', 'subset3 F1']
panel_b_labels = ['Subset1\n(F1)', 'Subset2\n(F1)', 'Subset3\n(F1)']

fig_width = 7.5
fig_height = 11.0
fig, axes = plt.subplots(4, 2, figsize=(fig_width, fig_height))

n_groups = len(GROUP_ORDER)
total_width = 0.75
bar_height = total_width / n_groups

def plot_panel(ax, metric_keys, metric_labels, mask_ratio_data, title, is_bottom_row):
    y_pos = np.arange(len(metric_labels))

    for i, group in enumerate(GROUP_ORDER):
        offset = (i - 1) * bar_height

        means = [mask_ratio_data[group][k][0] for k in metric_keys]
        stds = [mask_ratio_data[group][k][1] for k in metric_keys]

        ax.barh(
            y_pos + offset,
            means,
            bar_height,
            xerr=stds,
            color=GROUP_PALETTE[group],
            alpha=0.9,
            label=group if title.startswith("A") else "",
            capsize=1.5,
            error_kw={'linewidth': 0.6, 'elinewidth': 0.6},
            edgecolor='white',
            linewidth=0.5
        )

        for j, (mean, std) in enumerate(zip(means, stds)):
            text_x = min(mean + std + 0.01, 1.01)
            label_text = f'{mean:.3f}±{std:.3f}'
            ax.text(
                text_x,
                y_pos[j] + offset,
                label_text,
                va='center',
                ha='left',
                fontsize=5,
                color='black'
            )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(metric_labels, fontsize=7)
    ax.invert_yaxis()

    if is_bottom_row:
        ax.set_xlabel('Score', fontsize=8)

    ax.set_title(title, fontsize=9, fontweight='bold', loc='left', pad=8)
    ax.grid(True, alpha=0.2, linestyle='-', axis='x', linewidth=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 1.05)

mask_ratios = list(raw_data.keys())
panel_letters = [
    ['A', 'B'],
    ['C', 'D'],
    ['E', 'F'],
    ['G', 'H']
]

for row_idx, ratio_name in enumerate(mask_ratios):
    ax_left = axes[row_idx, 0]
    ax_right = axes[row_idx, 1]
    is_bottom = (row_idx == len(mask_ratios) - 1)

    plot_panel(
        ax_left,
        panel_a_keys,
        panel_a_labels,
        raw_data[ratio_name],
        f'{panel_letters[row_idx][0]}  {ratio_name} - Global Metrics',
        is_bottom
    )

    plot_panel(
        ax_right,
        panel_b_keys,
        panel_b_labels,
        raw_data[ratio_name],
        f'{panel_letters[row_idx][1]}  {ratio_name} - Subset Performance',
        is_bottom
    )

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.02),
    ncol=3,
    frameon=False,
    fontsize=8
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.07, hspace=0.38, wspace=0.25)

save_dir = Path('./')
save_dir.mkdir(parents=True, exist_ok=True)

output_pdf = save_dir / "Comparison_MaskRatios_with0.pdf"
plt.savefig(output_pdf, dpi=1200, bbox_inches='tight', transparent=True)

print(f"✅ 图表已生成: {output_pdf}")
plt.show()
